# ds004752 V5.6 Colab Full-Cohort Gate 0 Audit

Notebook nay duoc thiet ke de ban chay tu tren xuong duoi theo thu tu tung cell. Muc tieu la dong bo repo, xac nhan patch Gate 0 dang duoc dung, materialize full payload, chay full-cohort signal audit, va doc artifact closeout mot cach ro rang.

Notebook nay khong mo efficacy claim. No chi xac nhan data/signal readiness cho tranche V5.6 tiep theo.


## 1. Mount Google Drive

Artifacts va data duoc doc/ghi duoi `/content/drive/MyDrive/eeg-ds004752`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Clone hoac update repo vao runtime `/content`

Notebook dung repo trong `/content/eeg-ds004752` de chay code. Cell nay clone neu chua co, hoac fetch/pull neu da co.


In [ ]:
from pathlib import Path
import subprocess

REPO_DIR = Path('/content/eeg-ds004752')
REPO_URL = 'https://github.com/BrianNguyen29/eeg-ds004752.git'

def run(cmd, cwd=None):
    print('$', ' '.join(str(x) for x in cmd))
    subprocess.run(cmd, cwd=cwd, check=True)

if not REPO_DIR.exists():
    run(['git', 'clone', REPO_URL, str(REPO_DIR)])
else:
    run(['git', 'fetch', 'origin'], cwd=REPO_DIR)
    run(['git', 'checkout', 'main'], cwd=REPO_DIR)
    run(['git', 'pull', '--ff-only', 'origin', 'main'], cwd=REPO_DIR)


## 3. Xac nhan repo dang o commit moi va patch Gate 0 da hien dien

Cell nay giup tranh truong hop artifact moi nhung runtime van dang dung code cu.


In [ ]:
%cd /content/eeg-ds004752
!git log --oneline -5
from pathlib import Path

text = Path('src/audit/gate0.py').read_text(encoding='utf-8')
checks = {
    'materialized_inventory': 'materialized_inventory' in text,
    'sample_signal_conclusion': 'sampled signal-level audit passed' in text,
    'ready_conclusion': 'primary cohort lock is ready' in text,
}
print(checks)
assert all(checks.values()), checks


## 4. Cai runtime Python va signal extras

Cell nay cai dependency can cho signal-level audit.


In [ ]:
%cd /content/eeg-ds004752
!INSTALL_SIGNAL_EXTRAS=1 bash bootstrap/install_runtime.sh
!python bootstrap/colab_quickstart.py


## 5. Sua apt dependency thieu tren Colab

Cell nay xu ly `netbase` va broken `dpkg` state truoc khi datalad/git-annex get payload.


In [ ]:
!sudo apt-get update -y
!sudo apt-get install -y netbase
!sudo apt-get install -f -y


## 6. Materialize toan bo ds004752 payload

Cell nay se lay full `.edf` va `.mat` payload vao Drive. Thoi gian chay co the dai.


In [ ]:
%cd /content/eeg-ds004752
!bash bootstrap/get_data_colab.sh /content/drive/MyDrive/eeg-ds004752/data all


## 7. Kiem tra nhanh mot vai file payload dai dien

Neu file da materialize dung, size se lon va khong con la pointer nho.


In [ ]:
!ls -lh /content/drive/MyDrive/eeg-ds004752/data/ds004752/sub-01/ses-01/eeg
!ls -lh /content/drive/MyDrive/eeg-ds004752/data/ds004752/sub-14/ses-08/ieeg
!ls -lh /content/drive/MyDrive/eeg-ds004752/data/ds004752/derivatives/sub-14/beamforming


## 8. Chay test va smoke check

Cell nay khong phai bang chung khoa hoc, nhung giup bat loi code/runtime som.


In [ ]:
%cd /content/eeg-ds004752
!python -m unittest tests.unit.test_gate0
!python -m src.cli smoke --profile t4_safe --config configs/data/snapshot_colab.yaml


## 9. Chay full-cohort Gate 0 signal audit

Cell nay tao Gate 0 run moi tren toan cohort voi `--include-signal`.


In [ ]:
%cd /content/eeg-ds004752
!python -m src.cli audit --profile t4_safe --config configs/data/snapshot_colab.yaml --include-signal --signal-max-sessions 68


## 10. Xac dinh thu muc Gate 0 moi nhat


In [ ]:
from pathlib import Path

latest = Path('/content/drive/MyDrive/eeg-ds004752/artifacts/gate0/latest.txt')
run_dir = Path(latest.read_text().strip())
print('run_dir =', run_dir)


## 11. Doc `manifest.json`, `cohort_lock.json`, `materialization_report.json`


In [ ]:
import json

manifest = json.loads((run_dir / 'manifest.json').read_text())
cohort_lock = json.loads((run_dir / 'cohort_lock.json').read_text())
materialization_report = json.loads((run_dir / 'materialization_report.json').read_text())
bridge_availability = json.loads((run_dir / 'bridge_availability.json').read_text())
audit_report = (run_dir / 'audit_report.md').read_text()

summary = {
    'run_dir': str(run_dir),
    'manifest_status': manifest['manifest_status'],
    'primary_eligibility_status': manifest['participants']['primary_eligibility_status'],
    'bridge_availability_status': manifest['derivatives']['bridge_availability_status'],
    'gate0_blockers': manifest['gate0_blockers'],
    'signal_status': manifest['signal_audit']['status'],
    'sessions_checked': manifest['signal_audit'].get('sessions_checked'),
    'mat_files_checked': manifest['signal_audit'].get('mat_files_checked'),
    'cohort_lock_status': cohort_lock['cohort_lock_status'],
    'n_primary_eligible': cohort_lock['n_primary_eligible'],
    'edf_materialized': materialization_report['payloads']['edf']['materialized_count'],
    'edf_total': materialization_report['payloads']['edf']['count'],
    'mat_materialized': materialization_report['payloads']['mat']['materialized_count'],
    'mat_total': materialization_report['payloads']['mat']['count'],
    'bridge_status_raw': bridge_availability['status'],
}

print(summary)


## 12. Assert Gate 0 closeout mong doi

Neu cell nay fail, dung lai va xem artifact truoc khi di tiep.


In [ ]:
assert summary['manifest_status'] == 'signal_audit_ready', summary
assert summary['primary_eligibility_status'] == 'signal_audit_ready', summary
assert summary['cohort_lock_status'] == 'signal_audit_ready', summary
assert summary['signal_status'] == 'ok', summary
assert summary['gate0_blockers'] == [], summary
assert summary['edf_materialized'] == summary['edf_total'], summary
assert summary['mat_materialized'] == summary['mat_total'], summary
assert summary['sessions_checked'] == 68, summary
assert summary['mat_files_checked'] == 15, summary
assert summary['bridge_availability_status'] == 'materialized_inventory', summary
assert summary['bridge_status_raw'] == 'materialized_inventory', summary
print('Gate 0 full-cohort signal audit assertions passed.')


## 13. In `audit_report.md`

Kiem tra phan Conclusion co phu hop voi trang thai signal-ready hien tai hay khong.


In [ ]:
print(audit_report[:5000])


## 14. Guard check cho `phase1_real`

Cell nay van duoc ky vong block cho den khi Gate 2.5 prereg da locked. No khong lien quan den viec Gate 0 da signal-ready.


In [ ]:
import subprocess

cmd = [
    'python', '-m', 'src.cli', 'phase1_real',
    '--profile', 'a100_fast',
    '--config', 'configs/prereg/prereg_bundle.json',
]
result = subprocess.run(cmd, capture_output=True, text=True)
print('returncode:', result.returncode)
if result.stdout:
    print(result.stdout)
if result.stderr:
    print(result.stderr)

if result.returncode == 0:
    raise RuntimeError('Guard unexpectedly opened; inspect prereg bundle state.')

print('Expected: real-data phase remains guarded.')
